# 马尔可夫决策过程（MDP）基础

马尔可夫决策过程（Markov Decision Process, MDP）是强化学习的数学基础，用于描述智能体在环境中如何进行序列决策。

---

## 1. MDP 的五元组定义

一个 MDP 通常写作：
$$
athcal{M}=(athcal{S},athcal{A},P,R,amma)
$$
其中：
- $athcal{S}$：状态集合（State）
- $athcal{A}$：动作集合（Action）
- $P(s'|s,a)$：状态转移概率
- $R(s,a,s')$：奖励函数
- $amma n [0,1)$：折扣因子

马尔可夫性含义：下一时刻仅依赖当前状态与动作，与更久远的历史无关。

## 2. 回报与价值函数

从时刻 $t$ 开始的累计折扣回报：
$$
G_t = um_{k=0}^{nfty} amma^k R_{t+k+1}
$$

状态价值函数与动作价值函数：
$$
V^{i}(s)=athbb{E}_{i}[G_t|S_t=s], uad Q^{i}(s,a)=athbb{E}_{i}[G_t|S_t=s,A_t=a]
$$

最优价值函数：
$$
V^*(s)=ax_{i} V^{i}(s), uad Q^*(s,a)=ax_{i} Q^{i}(s,a)
$$

## 3. Bellman 方程

Bellman 期望方程（针对策略 $i$）：
$$
V^{i}(s)=um_a i(a|s)um_{s'} P(s'|s,a)eft[R(s,a,s')+amma V^{i}(s')
ight]
$$

Bellman 最优方程：
$$
V^*(s)=ax_aum_{s'} P(s'|s,a)eft[R(s,a,s')+amma V^*(s')
ight]
$$

这就是值迭代、策略迭代等动态规划算法的理论基础。

In [ ]:
import numpy as np

# 一个 3 状态的小型 MDP：S0, S1, S2(终止态)
states = [0, 1, 2]
actions = [0, 1]  # 0: 保守, 1: 激进
gamma = 0.9

# 转移与奖励: T[s][a] = [(prob, next_state, reward), ...]
T = {
    0: {
        0: [(1.0, 1, 2.0)],
        1: [(1.0, 2, 5.0)],
    },
    1: {
        0: [(1.0, 2, 3.0)],
        1: [(1.0, 0, 0.0)],
    },
    2: {
        0: [(1.0, 2, 0.0)],
        1: [(1.0, 2, 0.0)],
    },
}

terminal_state = 2

In [ ]:
def value_iteration(T, states, actions, gamma=0.9, theta=1e-8, max_iter=1000):
    V = np.zeros(len(states), dtype=float)

    for _ in range(max_iter):
        delta = 0.0
        V_new = V.copy()

        for s in states:
            q_values = []
            for a in actions:
                q = 0.0
                for p, s_next, r in T[s][a]:
                    q += p * (r + gamma * V[s_next])
                q_values.append(q)

            V_new[s] = max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < theta:
            break

    policy = {}
    for s in states:
        q_values = []
        for a in actions:
            q = 0.0
            for p, s_next, r in T[s][a]:
                q += p * (r + gamma * V[s_next])
            q_values.append(q)
        policy[s] = int(np.argmax(q_values))

    return V, policy

In [ ]:
V_star, pi_star = value_iteration(T, states, actions, gamma=gamma)

print("最优状态价值 V*:")
for s in states:
    print(f"State {s}: {V_star[s]:.4f}")

print("\n最优策略 pi* (0:保守, 1:激进):")
for s in states:
    print(f"State {s}: Action {pi_star[s]}")

## 4. 与强化学习的关系

- 动态规划（DP）假设已知完整转移概率 $P$ 和奖励函数 $R$。
- 蒙特卡洛（MC）和时序差分（TD）方法通常在未知模型下，通过采样学习价值函数。
- Q-Learning 本质上是在逼近最优动作价值函数 $Q^*(s,a)$。

掌握 MDP + Bellman 方程，就理解了强化学习的大部分数学主线。